In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (rule 10)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Unique cmdb_id values (sorted lists)
metric_cmdb_ids = sorted(metric['cmdb_id'].dropna().unique().tolist())
trace_cmdb_ids = sorted(trace['cmdb_id'].dropna().unique().tolist())
log_cmdb_ids = sorted(log['cmdb_id'].dropna().unique().tolist())
error_logs_cmdb_ids = sorted(error_logs['cmdb_id'].dropna().unique().tolist())

# 2) Metric kpi_name counts (top 50)
metric_kpis_counts_df = (
    metric['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
    .head(50)
)
metric_kpis_counts = metric_kpis_counts_df.to_dict(orient='records')

# 3) Trace trace_name counts (top 50)
trace_names_counts_df = (
    trace['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
    .head(50)
)
trace_names_counts = trace_names_counts_df.to_dict(orient='records')

# 4) Log log_name counts (top 50)
log_names_counts_df = (
    log['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'count'})
    .head(50)
)
log_names_counts = log_names_counts_df.to_dict(orient='records')

# 5) Heads (up to first 10 rows) with selected columns; convert timestamps to ISO strings for JSON
def head_selected(df, cols, n=10):
    existing = [c for c in cols if c in df.columns]
    h = df.loc[:, existing].head(n).copy()
    if 'timestamp' in h.columns:
        h['timestamp'] = h['timestamp'].astype(str)
    return h.to_dict(orient='records')

metric_head = head_selected(metric, ['timestamp', 'cmdb_id', 'kpi_name', 'value'], 10)
trace_head = head_selected(trace, ['timestamp', 'cmdb_id', 'trace_name', 'value'], 10)
log_head = head_selected(log, ['timestamp', 'cmdb_id', 'log_name', 'value'], 10)
error_logs_head = head_selected(error_logs, ['timestamp', 'cmdb_id', 'message'], 10)

# Assemble final JSON-serializable dictionary
result = {
    'metric_cmdb_ids': metric_cmdb_ids,
    'trace_cmdb_ids': trace_cmdb_ids,
    'log_cmdb_ids': log_cmdb_ids,
    'error_logs_cmdb_ids': error_logs_cmdb_ids,
    'metric_kpis_counts': metric_kpis_counts,
    'trace_names_counts': trace_names_counts,
    'log_names_counts': log_names_counts,
    'metric_head': metric_head,
    'trace_head': trace_head,
    'log_head': log_head,
    'error_logs_head': error_logs_head
}

result

```
Out[1]:
```
Summary of the execution results:

1) Unique cmdb_id values
- metric.csv: ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db']
- trace.csv: [] (no trace cmdb_ids present)
- log.csv: ['carts', 'carts-db', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'session-db', 'shipping', 'user']
- error_logs.csv: [] (no error_logs cmdb_ids present)

2) metric.csv KPI name counts (top counts shown)
- Top counts (values only, corresponding KPI names available in the dataframe): [375, 375, 375, 200, 175, 175, 175, 50]

3) trace.csv trace_name counts
- None (trace_names_counts is empty)

4) log.csv log_name counts (top 50 shown)
- Top counts: [225, 225] (the data shows two log names with equal counts)

5) Example rows (up to first 10) — compact highlights
- metric examples (first rows): entries include
  - carts: cpu = 2.684167776684423, latency-50 = 0.0136873368881461, latency-90 = 0.0232326543077404, mem ≈ 2.12e8, socket = 11.0, workload ≈ 7.5401
  - carts-db: cpu ≈ 2.6657, diskio ≈ 1.7045e6, mem ≈ 7.5368e7, socket = 7.0
  (These are the sample metric rows shown; full dataframe contains the KPI names and counts.)
- trace examples: none (trace_head is empty)
- log examples (first rows):
  - carts-db: log.error_count = 0, log.row_count = 1
  - catalogue: log.error_count = 0, log.row_count = 98
  - front-end: log.error_count = 0, log.row_count = 1214
  - orders: log.error_count = 0, log.row_count = 72
  - payment: log.error_count = 0, log.row_count = 60
- error_logs examples: none (error_logs_head is empty)

Notes:
- Trace and error_logs files contain no rows / cmdb_ids in this dataset.
- The metric KPIs observed in sample rows include cpu, latency-50, latency-90, mem, socket, workload, diskio.
- The log names observed include log.error_count and log.row_count (these two have the top counts shown).

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_cmdb_ids': ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'], 'trace_cmdb_ids': [], 'log_cmdb_ids': ['carts', 'carts-db', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'session-db', 'shipping', 'user'], 'error_logs_cmdb_ids': [], 'metric_kpis_counts': [{'count': 375}, {'count': 375}, {'count': 375}, {'count': 200}, {'count': 175}, {'count': 175}, {'count': 175}, {'count': 50}], 'trace_names_counts': [], 'log_names_counts': [{'count': 225}, {'count': 225}], 'metric_head': [{'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'cpu', 'value': 2.684167776684423}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'latency-50', 'value': 0.0136873368881461}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'latency-90', 'value': 0.0232326543077404}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'mem', 'value': 212174093.47368425}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'socket', 'value': 11.0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts', 'kpi_name': 'workload', 'value': 7.540105263157895}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'kpi_name': 'cpu', 'value': 2.665664191150962}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'kpi_name': 'diskio', 'value': 1704528.9680298131}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'kpi_name': 'mem', 'value': 75368340.21052632}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'kpi_name': 'socket', 'value': 7.0}], 'trace_head': [], 'log_head': [{'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'log_name': 'log.error_count', 'value': 0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'carts-db', 'log_name': 'log.row_count', 'value': 1}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'catalogue', 'log_name': 'log.error_count', 'value': 0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'catalogue', 'log_name': 'log.row_count', 'value': 98}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'front-end', 'log_name': 'log.error_count', 'value': 0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'front-end', 'log_name': 'log.row_count', 'value': 1214}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'orders', 'log_name': 'log.error_count', 'value': 0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'orders', 'log_name': 'log.row_count', 'value': 72}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'payment', 'log_name': 'log.error_count', 'value': 0}, {'timestamp': '2024-01-20 14:36:00+00:00', 'cmdb_id': 'payment', 'log_name': 'log.row_count', 'value': 60}], 'error_logs_head': []}```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric and log DataFrames from the kernel.
# Ensure timestamp columns are datetime UTC (they should already be)
if metric['timestamp'].dtype.kind in ('i','f'):
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
if log['timestamp'].dtype.kind in ('i','f'):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 14:33:22", utc=True)
end   = pd.to_datetime("2024-01-20 15:03:22", utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (before filtering)
thr = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# Filter metric rows to the incident window
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# Merge thresholds into window rows
merged_window = metric_window.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')

# Mark exceedances
merged_window['exceed'] = merged_window['value'] > merged_window['p95_threshold']

# Aggregates per (cmdb_id, kpi_name) within the window
count_df = merged_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].size().reset_index(name='count_points_in_window')
exceed_count_df = merged_window.groupby(['cmdb_id','kpi_name'], dropna=False)['exceed'].sum().reset_index(name='count_points_exceeding_p95')
max_df = merged_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].max().reset_index(name='max_value_in_window')
earliest_df = (
    merged_window[merged_window['exceed']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_timestamp_exceeding_p95_in_window')
)

# Combine aggregates
agg_window = count_df.merge(exceed_count_df, on=['cmdb_id','kpi_name'], how='left') \
                     .merge(max_df, on=['cmdb_id','kpi_name'], how='left') \
                     .merge(earliest_df, on=['cmdb_id','kpi_name'], how='left')

# Merge full threshold list with agg_window to include KPIs not present in the window
metric_candidates = thr.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for groups not present in the window
metric_candidates['count_points_in_window'] = metric_candidates['count_points_in_window'].fillna(0).astype(int)
metric_candidates['count_points_exceeding_p95'] = metric_candidates['count_points_exceeding_p95'].fillna(0).astype(int)
metric_candidates['max_value_in_window'] = metric_candidates['max_value_in_window'].where(~metric_candidates['max_value_in_window'].isna(), None)

# Convert earliest timestamps to ISO strings or None
def to_iso_or_none(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).isoformat()

metric_candidates['earliest_timestamp_exceeding_p95_in_window'] = metric_candidates['earliest_timestamp_exceeding_p95_in_window'].apply(to_iso_or_none)

# Select and sort top 15 by count_points_exceeding_p95 desc, then by max_value_in_window desc
metric_candidates_selected = metric_candidates[[
    'cmdb_id','kpi_name','p95_threshold','count_points_in_window',
    'count_points_exceeding_p95','earliest_timestamp_exceeding_p95_in_window','max_value_in_window'
]].copy()

metric_candidates_selected = metric_candidates_selected.sort_values(
    by=['count_points_exceeding_p95','max_value_in_window'],
    ascending=[False, False]
).head(15)

# Prepare metric_anomaly_candidates as list of dicts
metric_anomaly_candidates = metric_candidates_selected.to_dict(orient='records')

# 2) Aggregate per-component totals over KPIs in the window using all KPI groups
component_totals_all = (
    metric_candidates
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_count_points_in_window=('count_points_in_window', 'sum'),
        total_count_points_exceeding_p95=('count_points_exceeding_p95', 'sum')
    )
)

component_anomaly_summary_df = component_totals_all.sort_values(
    by='total_count_points_exceeding_p95', ascending=False
).head(10)

component_anomaly_summary = component_anomaly_summary_df.to_dict(orient='records')

# 3) From log.csv: for the top 5 components by metric anomalies, aggregate log.row_count and log.error_count in the same window
top5_components = component_anomaly_summary_df['cmdb_id'].head(5).tolist()

log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end) & (log['cmdb_id'].isin(top5_components))].copy()

if not log_window.empty:
    sums = log_window.groupby(['cmdb_id','log_name'], dropna=False)['value'].sum().reset_index()
    pivot = sums.pivot(index='cmdb_id', columns='log_name', values='value').reset_index()
    counts = log_window.groupby('cmdb_id').size().reset_index()
    counts.columns = ['cmdb_id','number_of_log_rows_in_window']
    log_agg = pivot.merge(counts, on='cmdb_id', how='left')
    log_agg['sum_log_row_count'] = log_agg.get('log.row_count', 0).fillna(0).astype(float)
    log_agg['sum_log_error_count'] = log_agg.get('log.error_count', 0).fillna(0).astype(float)
    log_agg['number_of_log_rows_in_window'] = log_agg['number_of_log_rows_in_window'].fillna(0).astype(int)
    top_components_logs_df = log_agg[['cmdb_id','sum_log_row_count','sum_log_error_count','number_of_log_rows_in_window']]
else:
    top_components_logs_df = pd.DataFrame(columns=['cmdb_id','sum_log_row_count','sum_log_error_count','number_of_log_rows_in_window'])

# Ensure all top5 components are present with zeros if missing
if top5_components:
    missing = [c for c in top5_components if c not in top_components_logs_df['cmdb_id'].tolist()]
    if missing:
        add_rows = pd.DataFrame({
            'cmdb_id': missing,
            'sum_log_row_count': [0.0]*len(missing),
            'sum_log_error_count': [0.0]*len(missing),
            'number_of_log_rows_in_window': [0]*len(missing)
        })
        top_components_logs_df = pd.concat([top_components_logs_df, add_rows], ignore_index=True)

# Order rows as top5_components
top_components_logs_df['order'] = top_components_logs_df['cmdb_id'].apply(lambda x: top5_components.index(x) if x in top5_components else 999)
top_components_logs_df = top_components_logs_df.sort_values('order').drop(columns='order')

top_components_logs = top_components_logs_df.to_dict(orient='records')

# Final JSON-serializable result
result = {
    'metric_anomaly_candidates': metric_anomaly_candidates,
    'component_anomaly_summary': component_anomaly_summary,
    'top_components_logs': top_components_logs
}

result

```
Out[2]:
```
Summary of findings (plain English):

- Metric anomaly candidates (top 15):
  - Most anomalies are memory ("mem") KPIs across many services; each listed KPI had 25 points in the incident window and 2 points exceeding its global P95.
  - Notable entries (p95 → earliest exceeding time → max in window):
    - payment: diskio (p95 1.7913e10) → first exceed 2024-01-20T14:51:00+00:00 → max 1.8297e10
    - payment: mem (p95 6.2677e8) → first exceed 2024-01-20T14:57:00+00:00 → max 6.2914e8
    - catalogue-db: mem (p95 4.3056e8) → first exceed 2024-01-20T14:37:00+00:00 → max 4.3057e8
    - orders: mem → first exceed 2024-01-20T14:52:00+00:00 → max ~3.2724e8
    - shipping, queue-master, carts, front-end, rabbitmq, carts-db, user-db, orders-db, user, catalogue, session-db: mem (each with 2 exceedances; earliest exceeds between ~14:37 and 14:59 UTC)
  - All rows show count_points_exceeding_p95 = 2 for the listed KPIs.

- Component-level totals (top 10 by total exceedances):
  - carts: total_points_in_window 150, total_exceeding_p95 12
  - orders: 150 / 12
  - front-end: 175 / 12
  - shipping: 150 / 12
  - payment: 175 / 12
  - catalogue: 150 / 10
  - user: 175 / 10
  - session-db: 100 / 6
  - orders-db: 100 / 6
  - carts-db: 100 / 6

  (Top five by anomalies are: carts, orders, front-end, shipping, payment — each with 12 total exceedances.)

- Logs for the top 5 anomalous components (within the same incident window):
  - carts: sum_log_row_count = 1325.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 44
  - orders: sum_log_row_count = 2920.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - front-end: sum_log_row_count = 46836.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - shipping: sum_log_row_count = 1460.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - payment: sum_log_row_count = 2420.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50

- Quick interpretation:
  - The dominant signal is elevated memory usage (mem) across many services during the incident window, with consistent small numbers of exceedances per KPI (2 each).
  - The five components with the most aggregated metric exceedances are carts, orders, front-end, shipping, and payment.
  - Logs for those top-5 components show no recorded log.error_count during the window (all sum_log_error_count = 0.0), so errors in logs do not appear alongside the metric exceedances.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_anomaly_candidates': [{'cmdb_id': 'payment', 'kpi_name': 'diskio', 'p95_threshold': 17913296026.012657, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:51:00+00:00', 'max_value_in_window': 18296578838.12256}, {'cmdb_id': 'payment', 'kpi_name': 'mem', 'p95_threshold': 626767940.2666667, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:57:00+00:00', 'max_value_in_window': 629143641.0434783}, {'cmdb_id': 'catalogue-db', 'kpi_name': 'mem', 'p95_threshold': 430561034.23999995, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:37:00+00:00', 'max_value_in_window': 430565239.4666666}, {'cmdb_id': 'orders', 'kpi_name': 'mem', 'p95_threshold': 327106887.68, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:52:00+00:00', 'max_value_in_window': 327244049.06666666}, {'cmdb_id': 'shipping', 'kpi_name': 'mem', 'p95_threshold': 297190413.6533333, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:50:00+00:00', 'max_value_in_window': 297202346.6666667}, {'cmdb_id': 'queue-master', 'kpi_name': 'mem', 'p95_threshold': 257829683.2, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:51:00+00:00', 'max_value_in_window': 257833642.66666663}, {'cmdb_id': 'carts', 'kpi_name': 'mem', 'p95_threshold': 215347227.30666667, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:59:00+00:00', 'max_value_in_window': 215422530.7826087}, {'cmdb_id': 'front-end', 'kpi_name': 'mem', 'p95_threshold': 106886514.07719296, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:52:00+00:00', 'max_value_in_window': 107134429.86666666}, {'cmdb_id': 'rabbitmq', 'kpi_name': 'mem', 'p95_threshold': 94193063.25333333, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:48:00+00:00', 'max_value_in_window': 94227865.6}, {'cmdb_id': 'carts-db', 'kpi_name': 'mem', 'p95_threshold': 80996515.84, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:59:00+00:00', 'max_value_in_window': 81168651.13043478}, {'cmdb_id': 'user-db', 'kpi_name': 'mem', 'p95_threshold': 46299081.38666667, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:59:00+00:00', 'max_value_in_window': 46600370.08695652}, {'cmdb_id': 'orders-db', 'kpi_name': 'mem', 'p95_threshold': 40938946.56, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:59:00+00:00', 'max_value_in_window': 41057769.73913044}, {'cmdb_id': 'user', 'kpi_name': 'mem', 'p95_threshold': 8215579.306666667, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:59:00+00:00', 'max_value_in_window': 8222630.956521739}, {'cmdb_id': 'catalogue', 'kpi_name': 'mem', 'p95_threshold': 6071323.306666667, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:58:00+00:00', 'max_value_in_window': 6083450.434782608}, {'cmdb_id': 'session-db', 'kpi_name': 'mem', 'p95_threshold': 2943467.52, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:54:00+00:00', 'max_value_in_window': 2964343.466666667}], 'component_anomaly_summary': [{'cmdb_id': 'carts', 'total_count_points_in_window': 150, 'total_count_points_exceeding_p95': 12}, {'cmdb_id': 'orders', 'total_count_points_in_window': 150, 'total_count_points_exceeding_p95': 12}, {'cmdb_id': 'front-end', 'total_count_points_in_window': 175, 'total_count_points_exceeding_p95': 12}, {'cmdb_id': 'shipping', 'total_count_points_in_window': 150, 'total_count_points_exceeding_p95': 12}, {'cmdb_id': 'payment', 'total_count_points_in_window': 175, 'total_count_points_exceeding_p95': 12}, {'cmdb_id': 'catalogue', 'total_count_points_in_window': 150, 'total_count_points_exceeding_p95': 10}, {'cmdb_id': 'user', 'total_count_points_in_window': 175, 'total_count_points_exceeding_p95': 10}, {'cmdb_id': 'session-db', 'total_count_points_in_window': 100, 'total_count_points_exceeding_p95': 6}, {'cmdb_id': 'orders-db', 'total_count_points_in_window': 100, 'total_count_points_exceeding_p95': 6}, {'cmdb_id': 'carts-db', 'total_count_points_in_window': 100, 'total_count_points_exceeding_p95': 6}], 'top_components_logs': [{'cmdb_id': 'carts', 'sum_log_row_count': 1325.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 44}, {'cmdb_id': 'orders', 'sum_log_row_count': 2920.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'front-end', 'sum_log_row_count': 46836.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'shipping', 'sum_log_row_count': 1460.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'payment', 'sum_log_row_count': 2420.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}]}```
```

In [ ]:
# In[3]:

import pandas as pd
from itertools import product

# Reuse existing metric and log DataFrames already loaded in the kernel.
# Ensure timestamps are datetime UTC
if metric['timestamp'].dtype.kind in ('i','f'):
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
if log['timestamp'].dtype.kind in ('i','f'):
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Components and KPIs of interest
components = ["carts", "catalogue", "orders", "payment", "user"]
kpis = ["workload", "cpu"]

# Incident window (UTC)
start = pd.to_datetime("2024-01-20 14:33:22", utc=True)
end   = pd.to_datetime("2024-01-20 15:03:22", utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (no window filtering)
metric_subset = metric[metric['cmdb_id'].isin(components) & metric['kpi_name'].isin(kpis)].copy()

if not metric_subset.empty:
    p95_df = (
        metric_subset
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index(name='p95_threshold')
    )
else:
    p95_df = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_threshold'])

# Ensure all requested (component, kpi) pairs are present in the thresholds table
pairs = pd.DataFrame(list(product(components, kpis)), columns=['cmdb_id','kpi_name'])
p95_all = pairs.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# 1b) Filter metric rows to the incident window
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end) & metric['cmdb_id'].isin(components) & metric['kpi_name'].isin(kpis)].copy()

# Merge thresholds into window rows to evaluate exceedances
merged = metric_window.merge(p95_all, on=['cmdb_id','kpi_name'], how='right', suffixes=('','_thr'))

# For rows that came from threshold-only (no metric rows in window), there will be NaNs for value/timestamp
# Mark exceed where value > p95_threshold (will be False/NaN -> treat as False)
merged['exceed'] = False
mask_valid = merged['value'].notna() & merged['p95_threshold'].notna()
merged.loc[mask_valid, 'exceed'] = merged.loc[mask_valid, 'value'] > merged.loc[mask_valid, 'p95_threshold']

# Aggregates within window per (cmdb_id,kpi_name)
count_in_window = (
    metric_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .count()
    .reset_index(name='count_points_in_window')
)

exceed_count = (
    merged.groupby(['cmdb_id','kpi_name'], dropna=False)['exceed']
    .sum()
    .reset_index(name='count_points_exceeding_p95')
)

max_in_window = (
    metric_window.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .max()
    .reset_index(name='max_value_in_window')
)

earliest_exceed = (
    merged[merged['exceed']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_timestamp_exceeding_p95_in_window')
)

# Combine results with full pairs to ensure missing combos included
results = pairs.merge(p95_all, on=['cmdb_id','kpi_name'], how='left') \
               .merge(count_in_window, on=['cmdb_id','kpi_name'], how='left') \
               .merge(exceed_count, on=['cmdb_id','kpi_name'], how='left') \
               .merge(max_in_window, on=['cmdb_id','kpi_name'], how='left') \
               .merge(earliest_exceed, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts and max values appropriately
results['count_points_in_window'] = results['count_points_in_window'].fillna(0).astype(int)
results['count_points_exceeding_p95'] = results['count_points_exceeding_p95'].fillna(0).astype(int)
results['max_value_in_window'] = results['max_value_in_window'].where(~results['max_value_in_window'].isna(), None)

# Convert earliest timestamps to ISO strings or None
def to_iso_or_none(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).isoformat()
results['earliest_timestamp_exceeding_p95_in_window'] = results['earliest_timestamp_exceeding_p95_in_window'].apply(to_iso_or_none)

# Sort to present (optional): by count_points_exceeding_p95 desc then max_value_in_window desc
results_sorted = results.sort_values(by=['count_points_exceeding_p95','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)

# Keep only requested columns and limit to these 10 rows (5 components * 2 kpis = 10)
metric_out = results_sorted[[
    'cmdb_id','kpi_name','p95_threshold','count_points_in_window',
    'count_points_exceeding_p95','earliest_timestamp_exceeding_p95_in_window','max_value_in_window'
]].head(10).copy()

# 2) From log.csv, filter to incident window and compute per-component sums for the five components
log_window = log[(log['timestamp'] >= start) & (log['timestamp'] <= end) & (log['cmdb_id'].isin(components))].copy()

if not log_window.empty:
    # Sum values by cmdb_id and log_name
    sums = log_window.groupby(['cmdb_id','log_name'], dropna=False)['value'].sum().reset_index()
    pivot = sums.pivot(index='cmdb_id', columns='log_name', values='value').reset_index()
    # Count number of log rows per component in window
    counts = log_window.groupby('cmdb_id').size().reset_index(name='number_of_log_rows_in_window')
    logs_agg = pivot.merge(counts, on='cmdb_id', how='right').fillna(0)
else:
    logs_agg = pd.DataFrame({
        'cmdb_id': components,
        'number_of_log_rows_in_window': [0]*len(components)
    })

# Ensure columns exist and map to required output names; include all 5 components even if missing
if 'log.row_count' in logs_agg.columns:
    logs_agg['sum_log_row_count'] = logs_agg['log.row_count'].astype(float)
else:
    logs_agg['sum_log_row_count'] = 0.0
if 'log.error_count' in logs_agg.columns:
    logs_agg['sum_log_error_count'] = logs_agg['log.error_count'].astype(float)
else:
    logs_agg['sum_log_error_count'] = 0.0

# Ensure all components are present
logs_final = pd.DataFrame({'cmdb_id': components}).merge(
    logs_agg[['cmdb_id','sum_log_row_count','sum_log_error_count','number_of_log_rows_in_window']],
    on='cmdb_id',
    how='left'
).fillna({'sum_log_row_count':0.0,'sum_log_error_count':0.0,'number_of_log_rows_in_window':0})

# Convert number_of_log_rows_in_window to int
logs_final['number_of_log_rows_in_window'] = logs_final['number_of_log_rows_in_window'].astype(int)

# Prepare final JSON-serializable output
result = {
    'workload_cpu_anomalies': metric_out.to_dict(orient='records'),
    'candidate_logs': logs_final[['cmdb_id','sum_log_row_count','sum_log_error_count','number_of_log_rows_in_window']].to_dict(orient='records')
}

result

```
Out[3]:
```
Summary (plain English):

- Overall pattern
  - For each of the five components (carts, catalogue, orders, payment, user) both KPIs (workload and cpu) were evaluated.
  - Each reported KPI had 25 points in the incident window and 2 points exceeding its global P95 (value > P95).
  - Earliest exceedance timestamps and the max values in-window are shown below. Logs contain no recorded errors for these components during the window.

- Per-component KPI results (cmdb_id — KPI: p95_threshold → count_in_window / exceedances → earliest_exceed → max_in_window)
  - carts
    - workload: p95 = 8.558594927536232 → 25 pts / 2 exceed → earliest 2024-01-20T14:37:00+00:00 → max 8.566633333333334
    - cpu:      p95 = 2.617349831322688  → 25 pts / 2 exceed → earliest 2024-01-20T14:36:00+00:00 → max 2.700618854407268
  - catalogue
    - workload: p95 = 4.251896231884058  → 25 pts / 2 exceed → earliest 2024-01-20T14:37:00+00:00 → max 4.324416666666667
    - cpu:      p95 = 0.17875336298321298→ 25 pts / 2 exceed → earliest 2024-01-20T14:38:00+00:00 → max 0.1815374075250438
  - orders
    - workload: p95 = 2.15095           → 25 pts / 2 exceed → earliest 2024-01-20T14:47:00+00:00 → max 2.1655666666666664
    - cpu:      p95 = 2.0578601421875766→ 25 pts / 2 exceed → earliest 2024-01-20T14:40:00+00:00 → max 2.159814176525629
  - payment
    - workload: p95 = 2.1469833333333335→ 25 pts / 2 exceed → earliest 2024-01-20T14:37:00+00:00 → max 2.1689000000000003
    - cpu:      p95 = 23.42319364173063  → 25 pts / 2 exceed → earliest 2024-01-20T14:54:00+00:00 → max 23.91256161697328
  - user
    - workload: p95 = 19.042566666666666 → 25 pts / 2 exceed → earliest 2024-01-20T14:47:00+00:00 → max 19.323083333333333
    - cpu:      p95 = 1.0771162711989961 → 25 pts / 2 exceed → earliest 2024-01-20T14:38:00+00:00 → max 1.089851119254108

- Logs for these five components (same incident window)
  - carts:    sum_log_row_count = 1325.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 44
  - catalogue: sum_log_row_count = 3885.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - orders:   sum_log_row_count = 2920.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - payment:  sum_log_row_count = 2420.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50
  - user:     sum_log_row_count = 14119.0, sum_log_error_count = 0.0, number_of_log_rows_in_window = 50

- Brief interpretation
  - Both workload and cpu show small but consistent exceedances (2 points each) above their global P95 in the incident window for all five components.
  - No log.error_count was observed for these components in the window, so logs do not show corresponding error events.

The original code execution output of IPython Kernel is also provided below for reference:

{'workload_cpu_anomalies': [{'cmdb_id': 'payment', 'kpi_name': 'cpu', 'p95_threshold': 23.42319364173063, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:54:00+00:00', 'max_value_in_window': 23.91256161697328}, {'cmdb_id': 'user', 'kpi_name': 'workload', 'p95_threshold': 19.042566666666666, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:47:00+00:00', 'max_value_in_window': 19.323083333333333}, {'cmdb_id': 'carts', 'kpi_name': 'workload', 'p95_threshold': 8.558594927536232, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:37:00+00:00', 'max_value_in_window': 8.566633333333334}, {'cmdb_id': 'catalogue', 'kpi_name': 'workload', 'p95_threshold': 4.251896231884058, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:37:00+00:00', 'max_value_in_window': 4.324416666666667}, {'cmdb_id': 'carts', 'kpi_name': 'cpu', 'p95_threshold': 2.617349831322688, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:36:00+00:00', 'max_value_in_window': 2.700618854407268}, {'cmdb_id': 'payment', 'kpi_name': 'workload', 'p95_threshold': 2.1469833333333335, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:37:00+00:00', 'max_value_in_window': 2.1689000000000003}, {'cmdb_id': 'orders', 'kpi_name': 'workload', 'p95_threshold': 2.15095, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:47:00+00:00', 'max_value_in_window': 2.1655666666666664}, {'cmdb_id': 'orders', 'kpi_name': 'cpu', 'p95_threshold': 2.0578601421875766, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:40:00+00:00', 'max_value_in_window': 2.159814176525629}, {'cmdb_id': 'user', 'kpi_name': 'cpu', 'p95_threshold': 1.0771162711989961, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:38:00+00:00', 'max_value_in_window': 1.089851119254108}, {'cmdb_id': 'catalogue', 'kpi_name': 'cpu', 'p95_threshold': 0.17875336298321298, 'count_points_in_window': 25, 'count_points_exceeding_p95': 2, 'earliest_timestamp_exceeding_p95_in_window': '2024-01-20T14:38:00+00:00', 'max_value_in_window': 0.1815374075250438}], 'candidate_logs': [{'cmdb_id': 'carts', 'sum_log_row_count': 1325.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 44}, {'cmdb_id': 'catalogue', 'sum_log_row_count': 3885.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'orders', 'sum_log_row_count': 2920.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'payment', 'sum_log_row_count': 2420.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}, {'cmdb_id': 'user', 'sum_log_row_count': 14119.0, 'sum_log_error_count': 0.0, 'number_of_log_rows_in_window': 50}]}```
```